# Sales Order Lines: Bronze to Silver

This notebook is the development and inspection entry point. Production logic lives in `pipelines.sales_order_lines_silver`.


In [ ]:
from pathlib import Path
import os, sys, uuid
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "product_analysis" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name != "product_analysis":
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pyspark.sql import SparkSession
from pipelines.bronze_to_silver_common import configure_logging
from pipelines.sales_order_lines_silver import (SalesOrderLinesSilverConfig, load_bronze_business_schema, create_sales_order_lines_tables, build_sales_order_lines_bronze_stream, build_sales_order_lines_batch_processor)
logger = configure_logging("INFO")
config = SalesOrderLinesSilverConfig(dataset_name="sales_order_lines", bronze_path="hdfs:///capstone/warehouse/bronze/sales_order_lines", checkpoint_path="hdfs://localhost:9000/capstone/checkpoints/bronze_to_silver/sales_order_lines", silver_table="datalake.silver.sales_order_lines", rejected_table="datalake.quarantine.sales_order_lines_rejected", audit_table="datalake.operations.silver_ingestion_audit", exploration_table="datalake.operations.explorations", expectation_table="datalake.operations.expectation_profiles", schema_file=str(PROJECT_ROOT / "schemas" / "sales_order_lines_schema.json"), pipeline_run_id=f"sales-order-lines-{uuid.uuid4()}", max_files_per_trigger=5)


In [ ]:
builder = (SparkSession.builder.appName("sales_order_lines_bronze_to_silver").master(os.environ.get("SPARK_MASTER", "local[2]")).config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions").config(f"spark.sql.catalog.{config.catalog}", "org.apache.iceberg.spark.SparkCatalog").config(f"spark.sql.catalog.{config.catalog}.type", "hadoop").config(f"spark.sql.catalog.{config.catalog}.warehouse", os.environ.get("ICEBERG_WAREHOUSE", "hdfs:///capstone/warehouse/iceberg")).config("spark.sql.session.timeZone", "UTC").config("spark.sql.caseSensitive", "true"))
if os.environ.get("ICEBERG_SPARK_RUNTIME_JAR"): builder = builder.config("spark.jars", os.environ["ICEBERG_SPARK_RUNTIME_JAR"])
spark = builder.getOrCreate()


In [ ]:
business_schema = load_bronze_business_schema(config.schema_file)
create_sales_order_lines_tables(spark, config)
batch_processor = build_sales_order_lines_batch_processor(config=config, logger=logger)


In [ ]:
bronze_stream = build_sales_order_lines_bronze_stream(spark, config, business_schema)
query = (bronze_stream.writeStream.queryName("sales_order_lines_bronze_to_silver").foreachBatch(batch_processor).option("checkpointLocation", config.checkpoint_path).trigger(availableNow=True).start())
query.awaitTermination()
if query.exception() is not None: raise RuntimeError(query.exception())


In [ ]:
for table in (config.silver_table, config.rejected_table, config.audit_table, config.exploration_table, config.expectation_table):
    if spark.catalog.tableExists(table): spark.catalog.refreshTable(table)
spark.table(config.silver_table).groupBy().count().show()
spark.table(config.audit_table).where(f"pipeline_run_id = '{config.pipeline_run_id}'").show(truncate=False)
